# End-to-end извлечение: Nougat vs regex/ML-pipeline

Цель — проверить гипотезу «одна end-to-end модель вместо пайплайна»: `facebook/nougat-base` превращает целую страницу PDF в Markdown (текст + формулы в LaTeX + таблицы) за один проход. Если бы это работало на русских нормативах, оно бы заменило layout-детекцию + распознавание формул + парсер таблиц.

Ключевой вопрос: **Nougat-base обучен на английских научных статьях (arXiv).** Работает ли он на русскоязычном СП с кириллицей?

## Что сравниваем

- **Nougat-base** (end-to-end, одна модель) — на пилотных страницах.
- **Текущий regex-pipeline** (`extract_tables` / `extract_formulas`) — baseline.
- **DocLayout-YOLO + pix2text/TableTransformer** — слой за слоем (nb01, nb03).

Пилотные страницы: 15 (таблица 4.1), 28 (таблица + подпись), 38 (изолированная формула).

In [ ]:
from __future__ import annotations

import sys
import time
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import pdfplumber
import pymupdf
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import NougatProcessor, VisionEncoderDecoderModel

# Путь к корню репозитория (надёжно при запуске из любого cwd).
ROOT = Path.cwd().resolve()
while not (ROOT / "docs" / "normative-base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ml" / "src"))

from normatives_pipeline.parser.extractors import extract_formulas, extract_tables

PDF = ROOT / "docs" / "normative-base" / "СП 22.13330.2016.pdf"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("ROOT:", ROOT)
print("device:", DEVICE)


## Загрузка Nougat-base

`NougatProcessor` сам кадрирует поля, ресайзит и нормализует страницу (DoNuT-препроцессинг), а `VisionEncoderDecoderModel` — это Swin-энкодер + mBART-декодер.

In [ ]:
processor = NougatProcessor.from_pretrained("facebook/nougat-base")
nougat = VisionEncoderDecoderModel.from_pretrained("facebook/nougat-base")
nougat.to(DEVICE).eval()

n_params = sum(p.numel() for p in nougat.parameters()) / 1e6
print(f"Nougat-base: {n_params:.0f} M параметров")


## Хелперы: рендер страницы и инференс с замером времени/VRAM

In [ ]:
def render_page(page_no: int, dpi: int = 150) -> Image.Image:
    doc = pymupdf.open(PDF)
    page = doc[page_no - 1]
    pix = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csRGB)
    doc.close()
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def nougat_image(img: Image.Image, max_length: int = 2048) -> tuple[str, float, float]:
    pixel_values = processor(images=img, return_tensors="pt").pixel_values.to(DEVICE)
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.no_grad():
        out = nougat.generate(
            pixel_values,
            min_length=1,
            max_length=max_length,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
        )
    dt = time.time() - t0
    gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0
    raw = processor.batch_decode(out, skip_special_tokens=True)[0]
    return processor.post_process_generation(raw, fix_markdown=False), dt, gb


## Прогон Nougat на русских страницах

In [ ]:
for pno, why in [(15, "таблица 4.1"), (28, "таблица + подпись"), (38, "изолированная формула")]:
    img = render_page(pno)
    md_text, dt, gb = nougat_image(img)
    shown = repr(md_text[:300]) if md_text.strip() else "(пусто)"
    print(f"===== стр. {pno} ({why}) | {dt:.1f}s | {gb:.2f} GB =====")
    print(shown)
    print()


**Наблюдение:** на всех русских страницах Nougat выдаёт пустую строку (декодер почти сразу выдаёт EOS). Это не ошибка загрузки — модель проверена ниже на контрольном англоязычном изображении.

## Контроль: Nougat на синтетическом англоязычном изображении

In [ ]:
img = Image.new("RGB", (1200, 400), "white")
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 28)
except OSError:
    font = ImageFont.load_default()
draw.text((40, 40), "The equation is $E = mc^2$.", fill="black", font=font)
draw.text((40, 120), "Table 1: results", fill="black", font=font)

md_text, dt, gb = nougat_image(img, max_length=1024)
print(f"[{dt:.1f}s] {md_text!r}")


**Наблюдение:** на английском Nougat корректно восстанавливает текст и формулу в LaTeX-разметке. Значит модель загружена правильно, а пустота на СП объясняется тем, что модель не обучалась на кириллице и русском нормативном стиле.

## Baseline: что даёт текущий regex-pipeline на тех же страницах

Для контраста — `extract_tables` и `extract_formulas` из текущего пайплайна.

In [ ]:
tables = extract_tables(PDF, max_pages=60)
print("regex tables (первые 60 стр.):", len(tables))
for t in tables:
    if t.page in (15, 28):
        print(f"--- стр. {t.page}, таблица {t.number}, строк: {len(t.rows)}")
        for row in t.rows[:3]:
            print("   ", row)

with pdfplumber.open(PDF) as pdf:
    pages_text = [p.extract_text() or "" for p in pdf.pages[:60]]

formulas = extract_formulas(pages_text)
print("regex formulas (первые 60 стр.):", len(formulas))
for f in formulas:
    if f.page == 38:
        print(f"--- стр. {f.page}, формула {f.number}: {f.raw_text[:120]!r}")


## Выводы

- **Nougat-base не применим «как есть»** к русским нормативам: на кириллических страницах даёт пустой вывод, хотя на английском работает корректно. Причина — домен обучения (arXiv, англ., научные статьи).
- **End-to-end в один проход в MVP не заходит**: чтобы Nougat заработал на русском, нужен fine-tune на размеченном корпусе русских нормативных PDF — это дорого и требует датасета, которого пока нет. Откладываем.
- **Слоистый подход (nb01/nb03) остаётся основным**: DocLayout-YOLO для layout, pix2text/texify для формул, TableTransformer для таблиц — каждая модель обучалась на своей задаче и не зависит от языка документа.
- Следующий шаг — количественно оценить качество формул (nb03) и сравнить pix2text vs texify, чтобы выбрать лучший распознаватель формул для интеграции.